---
title: "From the gazette to Markdown"
subtitle: "Retrieving a single DOF note with dofjson and nota2md"
date: 2026-07-17
# author:
#  - name: INGEOTEC
abstract: >
  This walkthrough shows how the LegalIA toolchain turns one note of the
  Diario Oficial de la Federación into clean Markdown. Using the gazette's
  open-data service we download the summary of every note published on one
  day, select a single note from that list, and build its Markdown from the
  official HTML — three steps, two small packages: dofjson and nota2md. The
  third package of the repository, dof2md, is only needed when a note exists
  solely as scanned page images; it has its own dedicated page.
---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INGEOTEC/LegalIA/blob/master/website/pages/tools.ipynb)

## Three packages, one pipeline

The LegalIA repository ships three Python packages, now published on PyPI,
that cover complementary stages of the same pipeline:

::: {.fronts}
::: {.front}
### dofjson {.unnumbered .unlisted}

A thin client for the DOF's open-data JSON service. It answers questions like
*which notes were published on this day?* and *what is the content of this
note?*
:::
::: {.front}
### nota2md {.unnumbered .unlisted}

Given the identifier of a note, produces its Markdown, choosing the best
available source: official HTML when it exists, OCR of the scanned pages
otherwise.
:::
::: {.front}
### dof2md {.unnumbered .unlisted}

Operates at the coarsest grain: converts a *complete edition*'s PDF (or a
set of scanned page images) to Markdown via OCR. See its own
[dedicated page](dof2md.ipynb) for a worked example.
:::
:::

It is worth stating upfront which of the three this tour actually requires.
For a modern note, the DOF publishes its full text as HTML, and the path from
identifier to Markdown involves only
[dofjson](https://pypi.org/project/dofjson/) (to talk to the service) and
[nota2md](https://pypi.org/project/nota2md/) (to convert the HTML).
[dof2md](https://pypi.org/project/dof2md/) never enters the picture; it is
the heavy artillery that `nota2md` borrows when a note predates the HTML era
and survives only as scanned images. Its own
[dedicated page](dof2md.ipynb) uses it directly, on its own.

The packages are available on PyPI. If you are running this notebook outside
the repository — on [Google Colab](https://colab.research.google.com/), for
instance — uncomment the following cell:

In [ ]:
# %pip install dofjson nota2md

## Step 1: every note of one day

The gazette's open-data service is organized around dates and identifiers.
`dofjson.get_notas` takes a date and returns the day's summary: one
list of notes per edition (morning, evening, extraordinary), each note a
dictionary of metadata. We ask for July 15, 2026:

In [ ]:
import datetime as dt

import dofjson

day = dt.date(2026, 7, 15)
notas = dofjson.get_notas(day)
{k: len(v) for k, v in notas.items() if isinstance(v, list)}

The morning edition claims 69 notes, but the raw listing includes stub
entries without a title that the DOF uses internally. The client provides a
helper that drops them and leaves a clean index of the day:

In [ ]:
notas = dofjson.quita_notas_sin_titulo(notas)
manana = notas["NotasMatutinas"]
len(manana)

Fifty-seven real notes. Each entry carries, among other fields, the note's
identifier (`codNota`), its title, the section of the gazette it appeared in
and whether the service holds its content as HTML (`existeHtml`). A glance at
the first few titles gives the flavor of an ordinary day in the gazette:

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "codNota": n["codNota"],
            "title": n["titulo"][:80] + ("…" if len(n["titulo"]) > 80 else ""),
            "HTML": n["existeHtml"],
        }
        for n in manana[:8]
    ]
)

## Step 2: selecting one note

Any note can now be addressed by its `codNota`. For this example we take the
agreement that sets the school calendars for the 2026--2027 school year — a
document that touches practically every household in the country:

In [ ]:
nota = next(n for n in manana if n["titulo"].startswith("Acuerdo número 07/07/26"))
cod_nota = nota["codNota"]
print(f"codNota   : {cod_nota}")
print(f"section   : {nota['codSeccion']}")
print(f"page      : {nota['pagina']}")
print(f"has HTML  : {nota['existeHtml']}")
print(f"title     : {nota['titulo'][:120]}…")

Because `existeHtml` is `"S"`, the service can hand us the note's full text
directly. `dofjson.get_nota` fetches that detail record; the HTML
lives in its `cadenaContenido` field. It is official, but it is not pretty:

In [ ]:
detalle = dofjson.get_nota(cod_nota)
print(detalle["cadenaContenido"][:400])

## Step 3: from HTML to Markdown

This is where `nota2md` takes over. Its single entry point,
`legal_provisions`, receives the `codNota`, an output directory and a
`source` argument that selects the conversion path: `"html"` (what we use
here), `"image"` or `"pdf"` for the OCR routes, or `"auto"` to let the
package decide — it prefers HTML whenever the note has it. The function
fetches whatever it needs through `dofjson` on its own; passing the day's
index we already downloaded merely saves a request:

In [ ]:
from pathlib import Path

from nota2md import legal_provisions

md_path = legal_provisions(cod_nota, Path("output"), source="html", notas_del_dia=notas)
md_path

The result is a single self-contained Markdown file, with the title as a
heading and the body cleaned of the gazette's presentational markup:

In [ ]:
print(md_path.read_text(encoding="utf-8")[:1200])

The same round trip is available from the command line, which is often more
convenient for batch work — the two commands below reproduce this whole
walkthrough:

```bash
dofjson 2026-07-15 --outdir output     # -> output/15072026-notas.json
nota2md 5793645 --outdir output        # -> output/nota-5793645.md
```

## When there is no HTML: the scanned decades

The HTML path covers the recent decades, but the archive reaches back to 1917
— and, as the [exploration of the titles](titles.ipynb) showed,
most of that history predates the electronic gazette. For those notes
`existeHtml` is `"N"` and the service offers only the scanned images of the
printed pages. `nota2md` handles this case with the same entry point: called
with `source="image"` (or `"pdf"`), it downloads the note's page images,
determines which pages the note spans, and runs them through an OCR pipeline
before cutting out the note's own text. That OCR machinery is not its own —
it is borrowed, lazily, from `dof2md`, which has
its own [dedicated page](dof2md.ipynb).

## Methodological note {.unnumbered .appendix}

This page is a Jupyter notebook executed against the DOF's open-data service
(`https://sidof.segob.gob.mx`) at render time; the outputs shown correspond
to the note with `codNota` 5793645, published in the morning edition of
July 15, 2026. The packages are documented in their respective READMEs in
the
[repository](https://github.com/INGEOTEC/LegalIA); none of them requires the
raw local archive used by the explorations.

The three packages this page describes — `dofjson`, `nota2md` and `dof2md`
— were written by the LegalIA team together with Claude, Anthropic's coding
assistant, through [Claude Code](https://claude.com/claude-code): the
assistant helped design and implement the open-data client, the
HTML-to-Markdown conversion and the scanned-page OCR pipeline, along with
their test suites. As with the rest of the project, the authors reviewed and
validated every change before it was committed.